> **These backends are unverified.**

> No clustrix cloud VM job (`provider="aws"`, `"gcp"`, `"azure"`, `"lambda"`, `"huggingface"`) has been shown to run end to end. Until recently the path could not have run at all: every cloud job died with a `KeyError` on its first line. That was fixed (issue #119), but nothing has since demonstrated a completed cloud job, and `scripts/collect_execution_evidence.py` does not cover these backends. This notebook describes the intended interface, not something that has been run.

> The backends that are verified working are `cluster_type="slurm"`, `cluster_type="ssh"` and `cluster_type="huggingface"` (HuggingFace Jobs, which is a different thing from the HuggingFace Spaces provider described here). See the Supported Cluster Types section of the documentation.


# Part 1: HuggingFace Jobs -- the verified backend

This is the part of this notebook that documents something that actually works: `cluster_type="huggingface"`, implemented in `clustrix/hf_jobs.py` (`HFJobsManager`). It has been run end to end against real HF Jobs containers. It has nothing to do with HuggingFace *Spaces* (web app hosting) -- that unrelated topic is documented separately as Part 2 below, under its original, unverified banner.

## Why this backend exists

HF Jobs runs a container, executes a command, and exits -- exactly Clustrix's model: hand over a function, run it, collect a result. It needs no cluster reservation, no VPN and no institutional SSH credentials, which is also why it is the substrate this project's own integration tests run against.

## Prerequisites

- `pip install huggingface_hub`
- An HF token with permission to run Jobs, either exported as `HF_TOKEN`, set in `configure(hf_token=...)`, or already on disk from `hf auth login`.

In [ ]:
from clustrix import configure, cluster

configure(
    cluster_type="huggingface",
    hf_username="your-hf-username",   # or hf_namespace= for an org
    # hf_token=...,                    # optional if HF_TOKEN is set, or `hf auth login` was run
    hf_flavor="cpu-basic",             # default; see the GPU section below before changing this
    hf_job_timeout="30m",              # default
)

@cluster(cores=1, memory="1GB")
def add(a, b):
    return a + b

# result = add(2, 3)  # requires a real HF token with Jobs access
# print(result)

## Behind the Scenes: How a Job Actually Runs

In order, from `HFJobsManager.submit_job` and `_bootstrap_source` in `clustrix/hf_jobs.py`:

1. The function, args and kwargs are packed with `dill` and base64-encoded.
2. **Payload staging.** HF rejects very large environment variables, so the encoded payload is capped at 256KB (`MAX_PAYLOAD_BYTES`). A payload under that limit travels in the `CLUSTRIX_PAYLOAD` env var. A larger one is uploaded to a private HF dataset repo (`<namespace>/clustrix-payloads` by default, or `hf_payload_repo`), and the job is instead given `CLUSTRIX_PAYLOAD_REPO`/`CLUSTRIX_PAYLOAD_FILE` plus a one-time `CLUSTRIX_HF_TOKEN` **secret** (not an env var) so it can download that one file. The staged file is deleted again once the job finishes, whether it succeeded or failed.
3. **Bootstrap.** The container runs a single `python -c "..."` bootstrap. Its first act is to `os.environ.pop('CLUSTRIX_HMAC_KEY')` -- the per-job signing key is removed from the environment *before* `pip install` runs, because a malicious or merely misbehaving package's own install hooks must not be able to read it. Only then does it `pip install` `dill`, `cloudpickle`, and anything named in `cluster_packages` / mirrored from your local environment (via `replicate_local_environment`, on by default -- the container starts from a bare Python image, so a function that imports `numpy` needs that mirrored or it fails only in the container).
4. The function is unpickled (`dill`, falling back to `cloudpickle`) and called. Its result -- or, on an exception, the error message, traceback, and (if picklable) the exception object itself -- is `dill`-serialized, HMAC-SHA256'd with the now-popped key, base64-encoded, and printed between marker lines (`---CLUSTRIX-RESULT-BEGIN---`/`...-END---`, or the `ERROR` equivalents).
5. **This side** polls the job, fetches its logs, and picks the *last* block in the log whose HMAC verifies against the per-job key -- not the first one it finds. A function is free to print anything, including a line that happens to equal a marker; only a verified tag distinguishes the real result from a decoy or from ordinary program output. An unverifiable or absent result raises `RuntimeError` rather than returning the log itself.
6. A function that raised is re-raised locally as the *original exception type* when it was picklable, with the remote traceback attached to the message. The container itself still exits `0` on a caught exception -- only a failure to *report* the exception is treated as a job failure -- so an ordinary `ValueError` from your function does not also trigger an HF "job failed" email to the account owner.

## GPU flavors bill real money

`is_gpu_flavor()` treats anything **not** prefixed `cpu-` as a GPU tier, and `hf_flavor` values like `a10g-small`, `a100-large`, `t4-medium` bill by the second for as long as the job runs. Requesting one without opting in raises `ValueError`:

In [ ]:
# Without the opt-in, this raises ValueError before anything is submitted:
configure(cluster_type="huggingface", hf_username="your-hf-username", hf_flavor="a10g-small")

@cluster(cores=4, memory="16GB")
def gpu_check():
    import torch
    return torch.cuda.is_available()

try:
    # gpu_check()
    pass
except ValueError as e:
    print(e)  # "Flavor 'a10g-small' is a GPU flavor and bills by the second. ..."

# Confirming you intend to pay for GPU time:
configure(
    cluster_type="huggingface",
    hf_username="your-hf-username",
    hf_flavor="a10g-small",
    hf_allow_gpu_flavors=True,   # required, on purpose -- this is a real-money gate
)
# result = gpu_check()  # now billed by the second for as long as this job runs

## Summary (Part 1)

- `cluster_type="huggingface"` is a verified backend: it has been run against real HF Jobs containers.
- Payloads under 256KB travel as an env var; larger ones stage through a private dataset repo, cleaned up afterward.
- Results are HMAC-verified before being deserialized, using a key that is removed from the environment before your code -- and before `pip install`'s own hooks -- can run.
- GPU flavors cost real money and require `hf_allow_gpu_flavors=True`; CPU flavors (the default) do not.

---

# Part 2: HuggingFace Spaces -- a different, unrelated, unverified topic

Everything below this point is the *original* content of this notebook. It is about deploying Gradio/Streamlit apps to HuggingFace **Spaces** (a web app hosting product) that happen to `import clustrix` and, in production, would point its SSH backend at a separately-provisioned compute cluster. It does not exercise the HF Jobs backend documented in Part 1 at all, and none of its cluster-execution claims have been verified end to end -- see the warning below, which predates this Part 1/Part 2 split.

## What Part 2 actually was, and the verdict

The ~1400 words that used to fill the rest of this notebook were a
Gradio/Streamlit "app template" walkthrough: HuggingFace Space hardware-tier
pricing tables, `git clone`/`huggingface_hub` deployment snippets, a
secrets-management guide, and a troubleshooting FAQ. None of it is a clustrix
feature. Checking the code confirms it: there is no `clustrix.spaces`
module, no Space-creation API, no Gradio/Streamlit integration anywhere in
this package. The only thing "integrating" clustrix with a Space was
`import clustrix` at the top of an `app.py` -- true of any pip package, and
not something this documentation should present as a supported workflow.

**What is real:** if you host a Gradio or Streamlit app on a HuggingFace
Space and want *that app* to hand work off to a separate compute cluster,
`clustrix` is just a normal dependency inside it. Add it to `requirements.txt`,
`import clustrix`, and use one of the two verified backends from elsewhere
in these docs -- `configure(cluster_type="ssh", ...)` (see
:doc:`ssh_tutorial`) if you have a machine to point it at, or
`configure(cluster_type="huggingface", ...)` (Part 1, above) if you want the
Space itself to launch HF Jobs. Everything about *how* `@cluster` then
behaves -- order of operations, config resolution, what can go wrong -- is
documented once, correctly, in :ref:`execution-model` and :ref:`limitations`;
repeating a second, unverified copy of it here would only invite drift.

There is nothing else backend-specific to "HuggingFace Spaces" for clustrix
to document.